# Modul A · Kapitel 1.4 — Neuronale Netze

## Challenge: Baue ein Netz, das handgeschriebene Ziffern liest

**Lernziel:** Du kannst erklären, woraus ein neuronales Netz besteht — Schichten, ReLU,
Softmax — und du baust den Vorwärtsdurchlauf selbst, trainierst ein Netz und bewertest es
ehrlich.

Das Gerüst kennst du aus 1.2 und 1.3: Modellfunktion, Kostenfunktion, trainieren, anwenden.
**Neu ist nur die Modellfunktion.** Alles andere bleibt, wo es war.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage — kurz überlegen, gerne mit der Nachbarin / dem Nachbarn |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind **7 Challenges**.

---
## 0 · Setup

▶️ Führe diese zwei Zellen aus. Danach sind alle Werkzeuge geladen und die Bilder liegen bereit.

Die zweite Zelle lädt beim ersten Mal 11 MB aus dem Netz herunter (oder liest die Datei aus
`data/`, falls sie schon daneben liegt).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
np.set_printoptions(suppress=True, precision=3)

# Wir trainieren später bewusst nur wenige Runden — die Warnung dazu blenden wir aus.
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("Setup fertig ✔")

In [ ]:
def lade_mnist():
    """Lädt MNIST — aus dem Ordner `data/` oder, falls nicht da, einmalig aus dem Netz (11 MB)."""
    import urllib.request

    quelle = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    kandidaten = [
        Path("mnist.npz"),
        Path("data") / "mnist.npz",
        Path("..") / "data" / "mnist.npz",
        Path("challenges/neuronale-netze/data/mnist.npz"),
    ]
    for pfad in kandidaten:
        if pfad.exists():
            print(f"Gelesen: {pfad}")
            daten = np.load(pfad)
            break
    else:
        print("mnist.npz nicht gefunden — lade 11 MB herunter …")
        urllib.request.urlretrieve(quelle, "mnist.npz")
        daten = np.load("mnist.npz")
        print("Gespeichert als: mnist.npz")

    return daten["x_train"], daten["y_train"], daten["x_test"], daten["y_test"]


bilder_train, y_train, bilder_test, y_test = lade_mnist()

print(f"{len(bilder_train):,} Trainingsbilder, {len(bilder_test):,} Testbilder")

---
## 1 · Die Daten

📖 Wir arbeiten mit **MNIST** — dem berühmtesten Datensatz des Machine Learning. 70.000
handgeschriebene Ziffern, in den 1990ern gesammelt von Angestellten des US-Volkszählungsamts
und von amerikanischen Schülerinnen und Schülern.

| | |
|---|---|
| **60.000** Bilder | zum Trainieren |
| **10.000** Bilder | zum Testen — die sieht das Modell erst ganz am Ende |
| **28 × 28 Pixel** | je Bild, Graustufen von 0 (schwarz) bis 255 (weiß) |
| **10 Klassen** | die Ziffern 0 bis 9 |

Zwei Dinge sind neu gegenüber 1.2 und 1.3:

1. **Die Eingabe ist ein Bild**, keine einzelne Zahl. Gleich siehst du, dass das keinen
   Unterschied macht — ein Bild *ist* eine Tabelle aus Zahlen.
2. **Es gibt zehn mögliche Antworten**, nicht zwei. Das ist eine **Mehrklassen-Klassifikation**
   (*multiclass*).

Und eines ist geschenkt: **Der Train/Test-Split ist schon gemacht.** MNIST kommt fertig
getrennt, wir müssen nicht mehr selbst mischen und schneiden wie in 1.2.

▶️ Schauen wir uns an, was da genau geladen wurde:

In [ ]:
print(f"Trainingsbilder: {bilder_train.shape}   ← {len(bilder_train):,} Stück à 28 × 28 Pixel")
print(f"Trainingslabels: {y_train.shape}")
print(f"Testbilder:      {bilder_test.shape}")
print(f"Testlabels:      {y_test.shape}")
print()
print(f"Pixelwerte gehen von {bilder_train.min()} bis {bilder_train.max()}  (Datentyp: {bilder_train.dtype})")
print(f"Klassen: {np.unique(y_train)}")
print()
print("So oft kommt jede Ziffer im Trainingssatz vor:")
for ziffer, anzahl in zip(*np.unique(y_train, return_counts=True)):
    print(f"  {ziffer}: {anzahl:>5}  {'█' * (anzahl // 150)}")

📖 Die Ziffern kommen ungefähr gleich häufig vor — die 1 am häufigsten, die 5 am seltensten.
Das ist gut: Es gibt kein starkes Klassenungleichgewicht. Wäre eine Ziffer deutlich häufiger
vertreten, könnte das Modell dazu neigen, diese Ziffer auch häufiger vorherzusagen. Dann müssten
wir die Trainingsdaten entsprechend ausgleichen.

Merk dir eine Zahl für später: Die häufigste Ziffer im **Testsatz** macht **rund 11 %** aus.
Wer stur immer „1" rät, liegt also in 11 % der Fälle richtig. **Das ist unsere Baseline** —
alles, was nicht deutlich darüber liegt, hat nichts gelernt.

---
## 2 · Erst schauen, dann rechnen

📖 Wie in jedem Kapitel: erst hinschauen. Bei Bildern heißt das buchstäblich hinschauen —
aber auch einmal *hinter* das Bild.

Denn ein Graustufenbild ist nichts anderes als eine **Tabelle aus Zahlen**: 28 Zeilen,
28 Spalten, in jeder Zelle ein Wert von 0 (schwarz) bis 255 (weiß). Für den Computer gibt es
kein „Bild" — es gibt nur diese 784 Zahlen.

▶️ Links das Bild, rechts derselbe Ausschnitt als Zahlen:

In [ ]:
bild = bilder_train[0]

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 5.5))

links.imshow(bild, cmap="gray")
links.set_title(f"Was wir sehen: eine {y_train[0]}")
links.axis("off")

# Rechts: der 8x8-Ausschnitt aus der Bildmitte, mit den echten Pixelwerten beschriftet
ausschnitt = bild[10:18, 8:16]
rechts.imshow(ausschnitt, cmap="gray")
for zeile in range(ausschnitt.shape[0]):
    for spalte in range(ausschnitt.shape[1]):
        wert = ausschnitt[zeile, spalte]
        rechts.text(spalte, zeile, str(wert), ha="center", va="center", fontsize=8,
                    color=ORANGE if wert < 128 else "#1f2937", fontweight="bold")
rechts.set_title("Was der Computer sieht: Zahlen")
rechts.axis("off")

plt.tight_layout()
plt.show()

print(f"Ein Bild hat die Form {bild.shape} — das sind {bild.size} Zahlen.")

### 🛠️ Challenge 1 — Das Ziffern-Raster

Jetzt du: Zeichne die **ersten 30 Trainingsbilder** in ein Raster aus 3 Zeilen und 10 Spalten.
Über jedes Bild soll sein Label stehen — also die Ziffer, die laut Datensatz dort steht.

Warum das lohnt: Du siehst sofort, wie unterschiedlich Menschen schreiben. Genau diese
Bandbreite muss unser Netz gleich verkraften.

*Tipp: `axes.flat` läuft der Reihe nach durch alle 30 Teilbilder. Ein Bild zeichnest du mit
`ax.imshow(bilder_train[i], cmap="gray")`, die Achsen blendest du mit `ax.axis("off")` aus, und
die Überschrift setzt `ax.set_title(str(y_train[i]), fontsize=9)`.*

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4.2))

...

plt.show()

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
fig, axes = plt.subplots(3, 10, figsize=(12, 4.2))

for i, ax in enumerate(axes.flat):
    ax.imshow(bilder_train[i], cmap="gray")
    ax.set_title(str(y_train[i]), fontsize=9)
    ax.axis("off")

plt.show()
```

Schau dir die Vielfalt an: geschlossene und offene Vieren, Einsen mit und ohne Fuß, Neunen, die
fast wie Vieren aussehen. **Es gibt keine Regel, die man von Hand aufschreiben könnte** — genau
deshalb lässt man ein Modell die Regel aus Beispielen lernen.

</details>

---
## 3 · Vorverarbeitung: aus Bildern werden Zahlenreihen

📖 In 1.2 und 1.3 hatten wir pro Haus **eine** Zahl (die Wohnfläche). Jetzt haben wir pro Bild
**784**. Am Verfahren ändert das nichts, an der Form der Daten schon. Zwei Handgriffe:

**1. Normalisieren** — Pixelwerte von 0–255 auf **0–1** bringen, also durch 255 teilen.
Warum? Neuronale Netze rechnen mit Gewichten, die anfangs winzig sind. Kommen dann Zahlen wie
255 herein, werden die Zwischenergebnisse riesig und das Training wird instabil. Kleine, gleich
skalierte Eingaben sind die halbe Miete. (Wer das Bonus-Notebook zum Gradientenabstieg gemacht
hat, kennt den Effekt: ohne Skalierung explodiert das Training in wenigen Schritten.)

**2. Flach machen** (*flatten*) — aus dem 28 × 28-Quadrat wird **eine Zeile mit 784 Zahlen**.
Warum? Unser Netz hat keine Vorstellung von „oben" und „links" — es bekommt einfach 784 Eingänge.
Ob Pixel Nummer 300 über oder neben Pixel Nummer 301 lag, weiß es nicht.

> 💬 Ja, das wirft Information weg — nämlich die ganze **räumliche Struktur**. Genau dafür gibt
> es später die *Convolutional Neural Networks*. Für 97 % Trefferquote reicht die flache
> Variante trotzdem, und sie ist ehrlicher zu dem, was du schon kannst.

### 🛠️ Challenge 2 — Bilder in Zahlenreihen verwandeln

Baue aus `bilder_train` und `bilder_test` die Matrizen `x_train` und `x_test`:
**normalisiert** (Werte zwischen 0 und 1) und **flach** (eine Zeile je Bild, 784 Spalten).

*Tipp: `.reshape(-1, 784)` macht aus `(60000, 28, 28)` die Form `(60000, 784)` — die `-1` heißt
„rechne dir die Zeilenzahl selbst aus". Das Teilen durch `255.0` erledigt NumPy für alle Zahlen
auf einmal, ganz ohne Schleife.*

In [ ]:
# TODO 1: Trainingsbilder — flach machen und auf 0–1 normalisieren
x_train = ...

# TODO 2: dasselbe für die Testbilder
x_test = ...

print(f"x_train: {x_train.shape}   Werte von {x_train.min():.1f} bis {x_train.max():.1f}")
print(f"x_test:  {x_test.shape}   Werte von {x_test.min():.1f} bis {x_test.max():.1f}")

In [ ]:
# ✅ Selbsttest
assert x_train.shape == (60000, 784), "x_train muss die Form (60000, 784) haben"
assert x_test.shape == (10000, 784), "x_test muss die Form (10000, 784) haben"
assert 0.0 <= x_train.min() and x_train.max() <= 1.0, "Die Werte müssen zwischen 0 und 1 liegen"
assert np.isclose(x_train.max(), 1.0), "Der hellste Pixel sollte danach genau 1.0 sein"
assert np.isclose(x_train[0].sum() * 255, bilder_train[0].sum()), "Nur teilen, nichts abschneiden"
print("✅ Challenge 2 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
x_train = bilder_train.reshape(-1, 784) / 255.0

x_test = bilder_test.reshape(-1, 784) / 255.0
```

Die Reihenfolge ist egal — `reshape` und `/ 255.0` kommen sich nicht in die Quere.

Ein Blick auf die Größenordnung: `x_train` ist jetzt eine Tabelle mit 60.000 Zeilen und 784
Spalten, also **47 Millionen Zahlen**. Genau so sieht Machine Learning in echt aus: Der
allergrößte Teil der Arbeit ist, Daten in die richtige Form zu bringen.

</details>

---
## 4 · Das Modell: mehrere Schichten hintereinander

$$\text{Neuron} \;=\; \text{Aktivierungsfunktion}(w \cdot x + b)$$

Ein einzelnes Neuron hat pro Pixel genau **ein Gewicht**. Es kann damit nur sagen: *„Ist Pixel
407 hell, spricht das für eine 3."* — Beiträge addieren, mehr nicht.

Für Ziffern reicht das nicht. Eine 7 kann oben links oder oben rechts beginnen, mit oder ohne
Querstrich; eine 9 unterscheidet sich von einer 4 nur durch die geschlossene Schleife oben.
Solche Aussagen brauchen ein **„erst … dann …"**:

> *„Wenn oben eine geschlossene Schleife ist **und** unten ein gerader Strich, dann ist es eine 9."*

Genau das kann ein einzelnes $w \cdot x + b$ nicht ausdrücken — es kann Beiträge nur addieren,
nicht verknüpfen. Und genau dafür gibt es **Schichten**.

Ein **neuronales Netz** ist nichts anderes als **viele solche Neuronen** — nebeneinander zu
einer *Schicht*, und mehrere Schichten hintereinander. Die Ausgabe der einen Schicht ist die
Eingabe der nächsten.

Unser Netz hat drei Schichten:

| Schicht | Eingänge | Neuronen | Was sie tut | Parameter |
|---|---:|---:|---|---:|
| **Schicht 1** | 784 Pixel | 128 | sucht einfache Muster: Striche, Bögen, Ecken | 784·128 + 128 = 100.480 |
| **Schicht 2** | 128 | 64 | kombiniert diese Muster zu größeren Formen | 128·64 + 64 = 8.256 |
| **Ausgang** | 64 | 10 | entscheidet: welche Ziffer? | 64·10 + 10 = 650 |
| | | | **gesamt** | **109.386** |

In jeder Schicht passiert exakt dasselbe wie in Kapitel 1.2 — nur mit vielen $w$ und $b$
gleichzeitig, deshalb schreibt man es als Matrix:

$$z = x \cdot W + b \qquad\text{(dieselbe Gerade, 128-fach parallel)}$$

$$a = \text{ReLU}(z) \qquad\text{(die Aktivierungsfunktion)}$$

Die Gewichte $W$ und die Biase $b$ zusammen heißen **Parameter** — die 109.386 in der letzten
Spalte sind **alles**, was das Netz ausmacht. In 1.2 waren es zwei. Am Prinzip ändert das nichts.

### Die Aktivierungsfunktion: ReLU statt Sigmoid

📖 In 1.3 haben wir die Sigmoid-Funktion benutzt, um aus einer Zahl eine Wahrscheinlichkeit zu
machen. **Innerhalb** eines Netzes nimmt man heute fast immer eine andere, viel simplere
Funktion — die **ReLU** (*Rectified Linear Unit*):

$$\text{ReLU}(z) = \max(0,\ z)$$

Auf Deutsch: *Negatives wird zu null, Positives bleibt, wie es ist.* Mehr nicht.

Warum überhaupt eine Aktivierungsfunktion? Weil sie die **Nichtlinearität** ins Modell bringt — und **ohne die wäre das ganze Netz sinnlos.** Zwei Geraden
hintereinander ergeben wieder eine Gerade — man könnte alle Schichten zu einer einzigen
zusammenrechnen und wäre wieder bei Kapitel 1.2. Erst der Knick bei null macht aus dem Stapel
etwas, das mehr kann als eine Gerade.

Und warum ReLU statt Sigmoid? Weil sie schneller ist (ein Vergleich statt einer e-Funktion) und
weil ihre Steigung für positive Werte konstant 1 bleibt. Bei der Sigmoid wird die Steigung an den
Rändern fast null — dann kommt beim Gradientenabstieg kein Signal mehr durch die hinteren
Schichten. Das ist das berüchtigte *vanishing gradient*-Problem, an dem neuronale Netze in den
1990ern jahrelang hängen blieben.

### 🛠️ Challenge 3 — Die ReLU

Schreibe die Funktion. Sie muss auch mit ganzen Arrays funktionieren, nicht nur mit einzelnen
Zahlen.

*Tipp: `np.maximum(a, b)` vergleicht elementweise und behält den größeren Wert — nicht zu
verwechseln mit `np.max()`, das nur das Maximum eines Arrays sucht.*

In [ ]:
def relu(z):
    """Negatives wird null, Positives bleibt."""
    # TODO: Ersetze die nächste Zeile durch die Formel
    raise NotImplementedError("Challenge 3: relu() implementieren")

In [ ]:
# ✅ Selbsttest
assert relu(3.0) == 3.0
assert relu(-3.0) == 0.0
assert relu(0.0) == 0.0
assert np.allclose(relu(np.array([-2.0, -0.1, 0.0, 0.5, 7.0])), [0.0, 0.0, 0.0, 0.5, 7.0])
print("✅ Challenge 3 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def relu(z):
    """Negatives wird null, Positives bleibt."""
    return np.maximum(0, z)
```

Eine Zeile — und das ist die Funktion, die in praktisch jedem modernen neuronalen Netz steckt,
von der Bilderkennung bis zum LLM. Manchmal in leicht abgewandelter Form (*GELU*, *SiLU*), aber
immer nach demselben Muster: unten abschneiden, oben durchlassen.

</details>

In [ ]:
# ▶️ So sieht sie aus — verglichen mit der Sigmoid aus Kapitel 1.3
z = np.linspace(-6, 6, 300)

fig, ax = plt.subplots()
ax.plot(z, relu(z), color=BLAU, linewidth=2.5, label="ReLU(z) = max(0, z)")
ax.plot(z, 1 / (1 + np.exp(-z)), color=GRAU, linewidth=2, linestyle="--",
        label="Sigmoid(z)  — Kapitel 1.3")
ax.axhline(0, color=GRAU, linewidth=1)
ax.axvline(0, color=GRAU, linewidth=1)

ax.set_xlabel("z")
ax.set_ylabel("Ausgabe")
ax.set_title("Der Knick bei null ist alles, was ReLU tut")
ax.legend()
plt.show()

### Die Ausgabe: Softmax statt Sigmoid

📖 Am Ende des Netzes stehen **10 Zahlen** — für jede Ziffer eine. Diese Zahlen (die *Scores*
oder *Logits*) können alles Mögliche sein: 8,2 oder −3,1 oder 0,04. Wir hätten aber gern
**10 Wahrscheinlichkeiten**, die sich zu 100 % addieren.

In 1.3 hat das die Sigmoid-Funktion erledigt — für **eine** Zahl. Die Verallgemeinerung auf
mehrere heißt **Softmax**:

$$\text{Softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{10} e^{z_j}}$$

In Worten: **erst „e hoch" auf jede Zahl anwenden, dann jede durch die Summe aller teilen.** Das
`e` sorgt dafür, dass alles positiv wird, und das Teilen dafür, dass die Summe genau 1 ergibt.

Ein Beispiel mit drei Klassen: aus den Scores $[2{,}0,\ 1{,}0,\ 0{,}1]$ wird
$[0{,}66,\ 0{,}24,\ 0{,}10]$. Die Reihenfolge bleibt, aber Abstände werden verstärkt — deshalb
*soft*max: eine weiche Version von „nimm einfach den größten".

### 🛠️ Challenge 4 — Die Softmax

Schreibe die Funktion. Sie bekommt die 10 Scores eines Bildes und gibt 10 Wahrscheinlichkeiten
zurück.

*Tipp: `np.exp(z)` rechnet die e-Funktion für alle Zahlen auf einmal, `np.sum(...)` summiert ein
Array zu einer Zahl. Achtung: Summiert wird über die **e-Werte**, nicht über `z`. Eine Zeile
reicht.*

In [ ]:
def softmax(z):
    """Macht aus beliebig vielen Scores genauso viele Wahrscheinlichkeiten, Summe = 1."""
    # TODO: Ersetze die nächste Zeile durch die Formel
    raise NotImplementedError("Challenge 4: softmax() implementieren")

In [ ]:
# ✅ Selbsttest
p = softmax(np.array([2.0, 1.0, 0.1]))
assert np.isclose(p.sum(), 1.0), "Die Wahrscheinlichkeiten müssen sich zu 1 addieren"
assert np.allclose(p, [0.659, 0.242, 0.099], atol=1e-3)
assert np.allclose(softmax(np.zeros(10)), 0.1), "Bei lauter gleichen Scores: 10 × 10 %"
assert softmax(np.array([0.0, 5.0, 1.0])).argmax() == 1, "Der größte Score muss vorne bleiben"
print("✅ Challenge 4 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def softmax(z):
    """Macht aus beliebig vielen Scores genauso viele Wahrscheinlichkeiten, Summe = 1."""
    return np.exp(z) / np.sum(np.exp(z))
```

Der Zusammenhang zu Kapitel 1.3 ist enger, als er aussieht: Für **zwei** Klassen ist die Softmax
exakt die Sigmoid-Funktion. Sie ist keine neue Idee, nur dieselbe für mehr als zwei Antworten.

*Kleine Warnung für den Eigenbau:* Bei sehr großen Scores wird $e^z$ astronomisch —
$e^{800}$ passt in keine Fließkommazahl mehr und ergibt `inf`. Bibliotheken ziehen deshalb vorher
den größten Wert ab: `np.exp(z - z.max())`. Am Ergebnis ändert das nichts (der Faktor kürzt sich
weg), an der Rechenbarkeit alles. Für unsere Zahlen ist beides in Ordnung.

</details>

### Alles zusammen: der Vorwärtsdurchlauf

📖 Jetzt hast du beide Bausteine — Zeit, das Netz zusammenzusetzen. Ein Bild wandert von vorn
nach hinten durch die Schichten, das nennt man den **Vorwärtsdurchlauf** (*forward pass*):

$$a_1 = \text{ReLU}(x \cdot W_1 + b_1) \qquad \text{784 Pixel} \rightarrow \text{128 Zahlen}$$
$$a_2 = \text{ReLU}(a_1 \cdot W_2 + b_2) \qquad \text{128} \rightarrow \text{64 Zahlen}$$
$$p = \text{Softmax}(a_2 \cdot W_3 + b_3) \qquad \text{64} \rightarrow \text{10 Wahrscheinlichkeiten}$$

Dreimal dieselbe Zeile. Das ist das komplette Modell — **jedes** neuronale Netz der Welt ist im
Kern diese Schleife, ChatGPT eingeschlossen. Dort sind es nur mehr Schichten und größere
Matrizen.

Das `@` in Python ist die **Matrixmultiplikation**: `x @ W` verrechnet alle 784 Eingaben mit
allen 128 Neuronen in einem Rutsch. Wer es ausschreiben will:
`z[k] = summe(x[i] * W[i, k] für alle i) + b[k]` — also 128-mal die Formel aus Kapitel 1.2.

### 🛠️ Challenge 5 — Der Vorwärtsdurchlauf

Schreibe die Funktion, die ein Bild durch alle drei Schichten schickt. Sie ist die
**Modellfunktion** dieses Kapitels — das Gegenstück zu `vorhersage()` aus 1.2 und
`wahrscheinlichkeit()` aus 1.3.

*Tipp: Genau die drei Formeln von oben, untereinander. Benutze deine eigenen Funktionen `relu()`
und `softmax()`.*

In [ ]:
def vorwaerts(x, W1, b1, W2, b2, W3, b3):
    """Schickt ein Bild (784 Zahlen) durchs Netz und gibt 10 Wahrscheinlichkeiten zurück."""
    # TODO 1: erste Schicht
    a1 = ...

    # TODO 2: zweite Schicht
    a2 = ...

    # TODO 3: Ausgabeschicht — hier kommt Softmax statt ReLU
    return ...

In [ ]:
# ✅ Selbsttest — ein Mini-Netz mit 2 Eingängen, bei dem man alles nachrechnen kann
W1_test = np.array([[1.0, 0.0], [0.0, -1.0]])
b1_test = np.array([0.0, 0.0])
W2_test = np.eye(2)
b2_test = np.zeros(2)
W3_test = np.zeros((2, 2))
b3_test = np.zeros(2)

# a1 = relu([1, -2]) = [1, 0];  a2 = [1, 0];  z3 = [0, 0]  →  softmax = [0.5, 0.5]
ergebnis = vorwaerts(np.array([1.0, 2.0]), W1_test, b1_test, W2_test, b2_test, W3_test, b3_test)
assert np.allclose(ergebnis, [0.5, 0.5]), "Bei lauter Nullen im Ausgang: 50 / 50"

W3_test = np.eye(2)   # jetzt zählt a2 = [1, 0] durch  →  softmax([1, 0])
ergebnis = vorwaerts(np.array([1.0, 2.0]), W1_test, b1_test, W2_test, b2_test, W3_test, b3_test)
assert np.allclose(ergebnis, [0.731, 0.269], atol=1e-3), "Der aktive Eingang muss gewinnen"
print("✅ Challenge 5 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def vorwaerts(x, W1, b1, W2, b2, W3, b3):
    """Schickt ein Bild (784 Zahlen) durchs Netz und gibt 10 Wahrscheinlichkeiten zurück."""
    a1 = relu(x @ W1 + b1)

    a2 = relu(a1 @ W2 + b2)

    return softmax(a2 @ W3 + b3)
```

Drei Zeilen. Vergleich das mit Kapitel 1.2 (`return w * x + b`) und 1.3
(`return sigmoid(w * x + b)`): Es ist dieselbe Zeile, dreimal gestapelt, mit Matrizen statt
einzelnen Zahlen.

Was hier **nicht** steht, ist genauso wichtig: Nirgends kommt vor, dass es um Ziffern geht. Diese
Funktion weiß nichts von Bildern. Sie multipliziert Zahlen. Das ganze „Wissen" steckt in den
Werten von $W$ und $b$ — und die haben wir noch nicht.

</details>

▶️ Probieren wir das Netz aus — mit **zufälligen** Gewichten, denn trainiert haben wir ja noch
nichts. Die Zelle unten **initialisiert** die Gewichte zufällig (die Biase auf null) — genau so
fängt auch jedes echte Training an — und schickt Bilder hindurch.

In [ ]:
# ▶️ Zufällige Initialisierung — der Zustand, in dem jedes Netz vor dem Training ist
zufall = np.random.default_rng(0)

W1 = zufall.normal(0, 0.1, (784, 128));  b1 = np.zeros(128)
W2 = zufall.normal(0, 0.1, (128, 64));   b2 = np.zeros(64)
W3 = zufall.normal(0, 0.1, (64, 10));    b3 = np.zeros(10)

p = vorwaerts(x_test[0], W1, b1, W2, b2, W3, b3)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 2]})
links.imshow(bilder_test[0], cmap="gray")
links.set_title(f"Tatsächlich: eine {y_test[0]}")
links.axis("off")

rechts.bar(range(10), p, color=GRAU)
rechts.bar([int(p.argmax())], [p.max()], color=ORANGE)
rechts.set_xticks(range(10))
rechts.set_xlabel("Ziffer")
rechts.set_ylabel("Wahrscheinlichkeit")
rechts.set_title(f"Das untrainierte Netz tippt auf {p.argmax()}")
plt.tight_layout()
plt.show()

# Und wie oft trifft es auf 1.000 Bildern?
alle = np.array([vorwaerts(bild, W1, b1, W2, b2, W3, b3) for bild in x_test[:1000]])
print(f"Trefferquote des Zufallsnetzes: {(alle.argmax(axis=1) == y_test[:1000]).mean():.1%}")

📖 Rund 10 % — genau so viel, wie blindes Raten bei zehn Möglichkeiten bringt. Das Netz ist
vollständig gebaut und rechnet fehlerfrei, es weiß nur nichts.

**Jetzt fehlt genau das, was in jedem Kapitel fehlt: gute Zahlen für $W$ und $b$.**

---
## 5 · Die Kostenfunktion: Cross-Entropy

📖 Dieselbe Frage: Wie messen wir, **wie falsch** das Netz liegt?

Damit man es als Formel schreiben kann, notiert man die richtige Antwort als **One-Hot-Vektor**:
zehn Zahlen, neunmal 0 und einmal 1 an der Stelle der richtigen Ziffer.

$$\text{Ziffer } 3 \;\rightarrow\; [0,\ 0,\ 0,\ \mathbf{1},\ 0,\ 0,\ 0,\ 0,\ 0,\ 0]$$

$$\text{Cross-Entropy} = -\frac{1}{n}\sum_{i=1}^{n}\ \sum_{k=0}^{9}\ y_{ik}\,\log(p_{ik})$$

Die innere Summe sieht nach viel aus, fällt aber sofort zusammen: Neun der zehn $y_{ik}$
sind null, es überlebt genau **ein** Summand — der für die richtige Ziffer.

▶️ Rechnen wir die Kosten für unser Zufallsnetz aus:

In [ ]:
# So sieht die richtige Antwort als One-Hot-Vektor aus
one_hot_vektor = np.eye(10)[y_test[:3]]
for i in range(3):
    print(f"Ziffer {y_test[i]}  →  {one_hot_vektor[i].astype(int)}")

print()

# Cross-Entropy = -log(Wahrscheinlichkeit der richtigen Ziffer), gemittelt
richtige = alle[np.arange(1000), y_test[:1000]]
kosten_zufall = -np.mean(np.log(richtige))

print(f"Cross-Entropy des Zufallsnetzes:  {kosten_zufall:.3f}")
print(f"Blind raten (1/10 für alles):     {-np.log(0.1):.3f}")
print(f"Perfektes Netz (1,0 für richtig): {abs(np.log(1.0)):.3f}")

**Training heißt: diese Zahl kleiner machen.** Und das Verfahren dafür ist dasselbe wie in jedem
Kapitel vorher der **Gradientenabstieg**.

<details>
<summary>💡 Für alle, die es genauer wollen: Was ist Backpropagation?</summary>

Beim Gradientenabstieg brauchst du für jeden der 109.386 Parameter die Antwort auf die Frage:
*„Wenn ich **diese** Zahl ein winziges bisschen ändere — werden die Kosten größer oder kleiner?"*

Bei zwei Parametern (Kapitel 1.2) rechnet man das direkt aus. Bei 109.386, die über drei
Schichten verteilt sind, wäre das aussichtslos — außer man geht **rückwärts**: erst der Ausgang,
dann Schicht 2, dann Schicht 1, und benutzt dabei jedes Zwischenergebnis wieder. Diese
Rückwärtsrechnung heißt **Backpropagation**; mathematisch ist sie nichts als die Kettenregel,
konsequent von hinten nach vorn angewendet.

Sie ist der Grund, warum große Netze überhaupt trainierbar sind — und sie ist der Grund, warum
die Aktivierungsfunktion nicht flach sein darf: Wo die Steigung null ist, kommt rückwärts kein Signal
mehr durch.

**Schreiben musst du sie nicht.** Das macht die Bibliothek, genau wie in 1.2 und 1.3.

</details>

---
## 6 · Trainieren

📖 Und jetzt der Moment, auf den alles hinausläuft. In 1.2 hieß es `LinearRegression()`, in 1.3
`LogisticRegression()`. Hier heißt es `MLPClassifier()` — *Multi-Layer Perceptron*, der
altmodische Name für „neuronales Netz mit mehreren Schichten".

Was dahinter passiert, hast du gerade selbst gebaut: Vorwärtsdurchlauf, Kosten messen, mit
Backpropagation die Gradienten holen, alle 109.386 Parameter ein Stück in die richtige Richtung
schieben. Und das viele tausend Mal.

▶️ Ausführen — das dauert ein paar Sekunden.

In [ ]:
from sklearn.neural_network import MLPClassifier

netz = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # zwei versteckte Schichten: 128 und 64 Neuronen
    activation="relu",              # deine Aktivierungsfunktion aus Challenge 3
    solver="adam",                  # Gradientenabstieg mit automatischer Schrittweite
    batch_size=128,                 # so viele Bilder pro Schritt
    max_iter=15,                    # 15 Epochen = 15-mal durch alle Trainingsbilder
    early_stopping=True,            # 10 % zurückhalten, um beim Training mitzumessen
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,                # damit alle im Raum dasselbe Netz bekommen
)

netz.fit(x_train, y_train)

print(f"Trainiert. {netz.n_iter_} Epochen gelaufen.")
print(f"Parameter: {sum(w.size for w in netz.coefs_) + sum(b.size for b in netz.intercepts_):,}")
print()
print("Form der gelernten Gewichte:")
for i, (W, b) in enumerate(zip(netz.coefs_, netz.intercepts_), start=1):
    print(f"  W{i}: {str(W.shape):>12}   b{i}: {str(b.shape):>8}")

📖 **Was die Einstellungen bedeuten** — das sind die Stellschrauben, an denen man in der Praxis
dreht:

| Einstellung | Bedeutung | Wenn man sie größer macht |
|---|---|---|
| `hidden_layer_sizes` | Anzahl und Größe der versteckten Schichten | mehr Kapazität, langsamer, mehr Overfitting-Gefahr |
| `max_iter` | **Epochen** — wie oft das Netz alle 60.000 Bilder sieht | besseres Training, irgendwann nur noch Auswendiglernen |
| `batch_size` | wie viele Bilder pro Lernschritt angeschaut werden | ruhigere, aber seltenere Schritte |
| `early_stopping` | hält 10 % der Trainingsdaten zurück, um beim Lernen mitzumessen | — |

**Wichtig zu `early_stopping`:** Diese 10 % sind **Validierungsdaten** und nicht dasselbe wie
unsere Testdaten. Die Testdaten fassen wir bis Abschnitt 8 nicht an. Man braucht beides: Die
Validierungsdaten helfen *während* des Trainings beim Nachjustieren — und genau deshalb taugen
sie danach nicht mehr als ehrliche Prüfung.

▶️ So sah das Training von innen aus:

In [ ]:
fig, (links, rechts) = plt.subplots(1, 2, figsize=(12, 4.5))

epochen = range(1, len(netz.loss_curve_) + 1)

links.plot(epochen, netz.loss_curve_, color=BLAU, linewidth=2, marker="o", markersize=4)
links.set_xlabel("Epoche")
links.set_ylabel("Cross-Entropy")
links.set_title("Die Kosten fallen")

rechts.plot(epochen, netz.validation_scores_, color=ORANGE, linewidth=2, marker="o", markersize=4)
rechts.set_xlabel("Epoche")
rechts.set_ylabel("Trefferquote")
rechts.set_title("Treffer auf den zurückgehaltenen 10 %")
rechts.set_ylim(0.9, 1.0)

plt.tight_layout()
plt.show()

print(f"Kosten   Epoche  1: {netz.loss_curve_[0]:.3f}   →   Epoche {len(netz.loss_curve_)}: {netz.loss_curve_[-1]:.3f}")
print(f"Treffer  Epoche  1: {netz.validation_scores_[0]:.1%}   →   Epoche {len(netz.validation_scores_)}: {netz.validation_scores_[-1]:.1%}")

💬 Die beiden Kurven erzählen unterschiedliche Geschichten. Links fallen die Kosten fleißig
weiter — von 0,35 auf 0,01. Rechts passiert ab Epoche 5 so gut wie nichts mehr; die Kurve
zappelt nur noch um 97,5 %. **Was heißt das?**

<details>
<summary>Antwort aufklappen</summary>

Das Netz wird auf den **Trainingsbildern** immer besser — es merkt sie sich zunehmend auswendig.
Auf **ungesehenen** Bildern bringt das ab einem gewissen Punkt nichts mehr.

Das ist **Overfitting**, live beim Entstehen. In Kapitel 1.2 war es nur eine Warnung: Ein Modell
aus zwei Parametern kann sich nichts merken. Hier sind es 109.386 bei 60.000 Bildern — jetzt
ist genug Platz zum Auswendiglernen da.

Deshalb ist die rechte Kurve die wichtigere. Sie ist der Grund, warum man beim Training
mitmisst, statt einfach möglichst lange zu trainieren: **Man will aufhören, bevor die orange
Kurve wieder fällt.** Genau das macht `early_stopping` automatisch.

</details>

In [ ]:
# ▶️ Was hat die erste Schicht gelernt? Jedes ihrer 128 Neuronen hat ein Gewicht je Pixel —
#    diese 784 Gewichte kann man wieder als 28 × 28-Bild anschauen.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))

for i, ax in enumerate(axes.flat):
    muster = netz.coefs_[0][:, i].reshape(28, 28)
    grenze = np.abs(muster).max()
    ax.imshow(muster, cmap="RdBu_r", vmin=-grenze, vmax=grenze)
    ax.set_title(f"Neuron {i}", fontsize=8)
    ax.axis("off")

fig.suptitle("16 der 128 Neuronen der ersten Schicht", fontweight="bold")
plt.tight_layout()
plt.show()

📖 Kein Neuron zeigt eine Ziffer. Man sieht **Striche, Bögen, helle und dunkle Flecken** — rot
heißt „hier spricht ein heller Pixel dafür", blau „hier spricht er dagegen". Jedes Neuron achtet
also auf ein Stück Form irgendwo im Bild.

Das ist der Unterschied zu einem einschichtigen Modell: Das hätte pro Ziffer genau **eine
Schablone** gelernt und geprüft, wie gut ein Bild dazu passt. Unser Netz hat **128 Teilmuster**,
die es anschließend frei kombinieren kann — und in der zweiten Schicht noch einmal 64
Kombinationen davon.

Genau darin liegt die Antwort auf die Frage von vorhin: *„geschlossene Schleife oben **und**
gerader Strich unten"* lässt sich so ausdrücken. Und niemand hat dem Netz gesagt, worauf es
achten soll — diese Muster sind aus 60.000 Beispielen und einer Kostenfunktion entstanden.

---
## 7 · Dein Netz im Einsatz

📖 Dein trainiertes Netz besteht aus genau **sechs Matrizen** — `W1, b1, W2, b2, W3, b3`, zusammen
109.386 Parameter. Mehr braucht es nicht. Und die Funktion, die daraus eine Vorhersage macht, hast du
in Challenge 5 selbst geschrieben.

▶️ Hier sind die gelernten Gewichte:

In [ ]:
W1, W2, W3 = netz.coefs_
b1, b2, b3 = netz.intercepts_

print(f"W1 {W1.shape}   b1 {b1.shape}")
print(f"W2 {W2.shape}    b2 {b2.shape}")
print(f"W3 {W3.shape}      b3 {b3.shape}")
print()
print(f"Kleinstes Gewicht: {min(W.min() for W in (W1, W2, W3)):.3f}")
print(f"Größtes Gewicht:   {max(W.max() for W in (W1, W2, W3)):.3f}")
print()
# Pixel 350 liegt mitten im Bild — hier stehen die Gewichte zu den ersten fünf Neuronen:
print("W1[350, :5] =", W1[350, :5])

### 🛠️ Challenge 6 — Die Postleitzahl auf dem Umschlag

📬 Genau dafür wurde MNIST gebaut: Die US-Post wollte Briefe automatisch sortieren.

Auf dem Umschlag unten stehen fünf handgeschriebene Ziffern. **Lies sie mit deinem eigenen
Vorwärtsdurchlauf** — nicht mit `netz.predict()`, sondern mit deiner `vorwaerts()`-Funktion aus
Challenge 5 und den gerade geladenen Gewichten.

Für jedes der fünf Bilder brauchst du:
1. die 10 Wahrscheinlichkeiten aus `vorwaerts(...)`,
2. die Ziffer mit der höchsten davon.

*Tipp: `np.argmax(p)` gibt dir den Index des größten Wertes — und weil die Ziffer 0 an Position
0 steht, die Ziffer 1 an Position 1 und so weiter, **ist** dieser Index bereits die Ziffer.
`p.max()` liefert die zugehörige Wahrscheinlichkeit.*

In [ ]:
# ▶️ Der Umschlag
UMSCHLAG = [2, 3, 5, 14, 15]   # fünf Bilder aus dem Testsatz

fig, axes = plt.subplots(1, 5, figsize=(9, 2.2))
for ax, i in zip(axes, UMSCHLAG):
    ax.imshow(bilder_test[i], cmap="gray")
    ax.axis("off")
fig.suptitle("Postleitzahl auf dem Umschlag", fontweight="bold")
plt.show()

In [ ]:
plz = ""

for i in UMSCHLAG:
    # TODO 1: die 10 Wahrscheinlichkeiten für das Bild x_test[i]
    p = ...

    # TODO 2: die Ziffer mit der höchsten Wahrscheinlichkeit
    ziffer = ...

    plz += str(ziffer)
    print(f"Bild {i:>3}:  gelesen als {ziffer}   (Sicherheit {p.max():.1%})")

print()
print(f"Die Postleitzahl lautet: {plz}")

In [ ]:
# ✅ Selbsttest
assert plz == "10115", f"Erwartet wird 10115, gelesen wurde {plz}"

# Und die Gegenprobe: liefert scikit-learn dasselbe wie deine eigene Funktion?
eigene = vorwaerts(x_test[UMSCHLAG[0]], W1, b1, W2, b2, W3, b3)
sklearn_p = netz.predict_proba(x_test[UMSCHLAG[0]].reshape(1, -1))[0]
assert np.allclose(eigene, sklearn_p, atol=1e-8), "Beide Wege müssen dasselbe liefern"

print(f"✅ Challenge 6 gelöst — {plz}, das ist Berlin-Mitte 🎉")
print(f"   Größte Abweichung zu netz.predict_proba(): {np.abs(eigene - sklearn_p).max():.2e}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
plz = ""

for i in UMSCHLAG:
    p = vorwaerts(x_test[i], W1, b1, W2, b2, W3, b3)

    ziffer = np.argmax(p)

    plz += str(ziffer)
    print(f"Bild {i:>3}:  gelesen als {ziffer}   (Sicherheit {p.max():.1%})")

print()
print(f"Die Postleitzahl lautet: {plz}")
```

**Das ist der Punkt dieses Kapitels.** Deine drei Zeilen aus Challenge 5 liefern exakt dieselben
Zahlen wie `netz.predict_proba()` — bis auf die letzte Nachkommastelle. `scikit-learn` macht beim
Vorhersagen nichts anderes als du: multiplizieren, ReLU, multiplizieren, ReLU, multiplizieren,
Softmax.

Der ganze Unterschied zwischen deinem Code und einer Bibliothek liegt im **Training** — und
selbst das ist „nur" Gradientenabstieg, effizient implementiert.

</details>

In [ ]:
# ▶️ Zum Spielen: Was hält das Netz von einem einzelnen Bild? Index frei wählen (0 … 9999).
INDEX = 1000

p = vorwaerts(x_test[INDEX], W1, b1, W2, b2, W3, b3)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 2]})
links.imshow(bilder_test[INDEX], cmap="gray")
links.set_title(f"Tatsächlich: {y_test[INDEX]}")
links.axis("off")

farben = [ORANGE if k == p.argmax() else GRAU for k in range(10)]
rechts.bar(range(10), p, color=farben)
rechts.set_xticks(range(10))
rechts.set_xlabel("Ziffer")
rechts.set_ylabel("Wahrscheinlichkeit")
rechts.set_title(f"Das Netz sagt: {p.argmax()}  ({p.max():.1%})")
plt.tight_layout()
plt.show()

for ziffer in np.argsort(-p)[:3]:
    print(f"  {ziffer}: {p[ziffer]:>7.2%}  {'█' * int(round(p[ziffer] * 40))}")

---
## 8 · Die ehrliche Prüfung: die Testdaten

📖 Jetzt — und erst jetzt — kommen die 10.000 Testbilder ins Spiel. Das Netz hat sie nie gesehen:
weder beim Training noch bei den 10 % zum Mitmessen. Sie sind die einzige ehrliche Antwort auf
die Frage, ob das Ding etwas taugt.

### 🛠️ Challenge 7 — Wie gut ist dein Netz wirklich?

Miss die Trefferquote auf den Testdaten.

1. Vorhersagen für alle 10.000 Testbilder berechnen.
2. Den Anteil der richtigen bestimmen.

*Tipp zu 1: Diesmal darfst du `netz.predict(x_test)` benutzen — deine eigene Funktion nimmt nur
ein Bild auf einmal, das wären 10.000 Durchläufe.*

*Tipp zu 2: `vorhersage_test == y_test` gibt eine Reihe aus `True`/`False`. Der Mittelwert
davon ist genau der Anteil der Treffer — `True` zählt als 1.*

In [ ]:
# TODO 1: Vorhersagen für alle Testbilder
vorhersage_test = ...

# TODO 2: Anteil richtiger Vorhersagen
treffer_test = ...

treffer_train = (netz.predict(x_train) == y_train).mean()

print(f"Trefferquote auf den Trainingsdaten: {treffer_train:.2%}")
print(f"Trefferquote auf den Testdaten:      {treffer_test:.2%}")
print(f"Fehler auf den Testdaten:            {int((1 - treffer_test) * len(y_test))} von {len(y_test):,}")

In [ ]:
# ✅ Selbsttest
assert len(vorhersage_test) == len(y_test), "Für jedes Testbild eine Vorhersage"
assert abs(treffer_test - 0.977) < 0.01, "Erwartet werden rund 97,7 %"
print(f"✅ Challenge 7 gelöst — {treffer_test:.2%} Treffer 🎉")
print()
print("Zum Vergleich:")
print(f"  immer '1' raten:       {(y_test == 1).mean():>7.2%}")
print(f"  dein neuronales Netz:  {treffer_test:>7.2%}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
vorhersage_test = netz.predict(x_test)

treffer_test = (vorhersage_test == y_test).mean()
```

97,7 % klingt nach viel. Bei Trefferquoten nahe 100 % ist aber die **Fehlerzahl** die
ehrlichere Größe: 230 falsch gelesene Ziffern. Der Unterschied zwischen 99 % und 99,5 %
halbiert die Arbeit der Nachkontrolle — das hört man der Prozentzahl nicht an.

Für die Poststelle von vorhin heißt es konkret: Bei fünf Ziffern je Postleitzahl liegt
$0{,}977^5 \approx 89\,\%$ — also wäre immer noch **jede neunte Adresse** falsch gelesen. Für
den echten Einsatz reicht das nicht; dafür braucht es die Modellgeneration, die am Ende dieses
Abschnitts steht.

</details>

💬 **Training 99,5 %, Test 97,7 % — in Kapitel 1.2 waren beide Zahlen gleich. Warum jetzt nicht
mehr?**

<details>
<summary>Antwort aufklappen</summary>

Weil das Modell groß genug geworden ist, um sich Dinge zu **merken**. Zwei Zahlen konnten sich
17.290 Häuser nicht merken. 109.386 Parameter können sich einen Teil von 60.000 Bildern sehr
wohl merken.

Die Lücke von 1,8 Prozentpunkten ist genau der Anteil, der auswendig gelernt statt verstanden
wurde. Solange sie klein bleibt, ist das kein Drama — man beobachtet sie. Wird sie groß
(Training 99,9 %, Test 85 %), hilft eines von dreien: **mehr Daten**, ein **kleineres Netz**,
oder **Regularisierung** — etwa der Parameter `alpha` im `MLPClassifier` oder *Dropout*, bei dem
in jedem Schritt zufällig ein Teil der Neuronen abgeschaltet wird, damit sich das Netz nicht auf
einzelne verlassen kann.

Das ist dieselbe Kennzahl, auf die man auch bei einem LLM mit Milliarden Parametern schaut. Nur
sind dort die Testdaten das eigentliche Problem: Bei einem Modell, das das halbe Internet gelesen
hat, ist schwer zu sagen, welcher Text wirklich ungesehen ist.

</details>

📖 Eine einzige Prozentzahl verschweigt aber wieder das Wichtigste: **welche** Fehler das Netz
macht. Bei zwei Klassen war das die Konfusionsmatrix aus Kapitel 1.3 mit ihren vier Feldern. Bei
zehn Klassen sind es 10 × 10 Felder — dieselbe Idee, nur größer.

▶️ Zeile = was tatsächlich dastand, Spalte = was das Netz gelesen hat.

In [ ]:
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_test, vorhersage_test)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.grid(False)
# log1p nur für die Farbe — sonst überstrahlt die Diagonale alle Fehler
ax.imshow(np.log1p(matrix), cmap="Blues")

for i in range(10):
    for j in range(10):
        if matrix[i, j] > 0:
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=8,
                    color="white" if i == j else "#1f2937",
                    fontweight="bold" if i != j else "normal")

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("… gelesen als")
ax.set_ylabel("tatsächlich eine …")
ax.set_title("Konfusionsmatrix: die Diagonale ist richtig, alles andere ein Fehler")
plt.tight_layout()
plt.show()

In [ ]:
# ▶️ Die häufigsten Verwechslungen
verwechslungen = [(i, j, matrix[i, j]) for i in range(10) for j in range(10)
                  if i != j and matrix[i, j] > 0]
verwechslungen.sort(key=lambda v: -v[2])

print("Am häufigsten vertut sich das Netz hier:")
for wahr, gelesen, anzahl in verwechslungen[:8]:
    print(f"  {wahr} gelesen als {gelesen}:  {anzahl:>3}-mal")

print()
print("Und so gut erkennt es die einzelnen Ziffern:")
for ziffer in range(10):
    quote = matrix[ziffer, ziffer] / matrix[ziffer].sum()
    print(f"  {ziffer}: {quote:>6.1%}  {'█' * int(round(quote * 40))}")

In [ ]:
# ▶️ Ein detaillierter Bericht je Ziffer — dieselben Kennzahlen wie in Kapitel 1.3,
#    nur zehnmal statt einmal.
from sklearn.metrics import classification_report

print(classification_report(y_test, vorhersage_test, digits=3,
                            target_names=[f"Ziffer {i}" for i in range(10)]))
print("Precision: Von allem, was das Netz als X gelesen hat — wie viel war wirklich X?")
print("Recall:    Von allen echten X — wie viele hat es gefunden?")
print("f1-score:  beides in einer Zahl zusammengefasst")

In [ ]:
# ▶️ Und jetzt das Interessanteste: die Fehler selbst
falsch = np.where(vorhersage_test != y_test)[0]
p_test = netz.predict_proba(x_test)

fig, axes = plt.subplots(3, 8, figsize=(12, 5.4))
for ax, i in zip(axes.flat, falsch[:24]):
    ax.imshow(bilder_test[i], cmap="gray")
    ax.set_title(f"{y_test[i]} → {vorhersage_test[i]}\n{p_test[i].max():.0%} sicher",
                 fontsize=8, color="#b91c1c")
    ax.axis("off")

fig.suptitle(f"{len(falsch)} Fehler insgesamt — hier die ersten 24", fontweight="bold")
plt.tight_layout()
plt.show()

💬 Schau dir diese Bilder ehrlich an: **Wie viele davon hättest du selbst richtig gelesen?**

<details>
<summary>Antwort aufklappen</summary>

Ein Teil ist tatsächlich unlesbar — abgeschnittene Striche, Vieren ohne Dach, Neunen ohne
Schleife. Je nach Studie machen Menschen auf MNIST zwischen 0,2 % und 2,5 % Fehler; unser Netz
liegt mit 2,3 % am unaufmerksamen Ende dieser Spanne. Für 15 Epochen und 109.386 Parameter ist das
nicht schlecht.

Zwei Dinge sind bemerkenswert:

* **Manche Fehler macht das Netz mit 99 % Sicherheit.** Die Wahrscheinlichkeit misst, wie gut
  ein Bild zu dem passt, was das Netz gelernt hat — nicht, ob es recht hat. Genau derselbe Punkt
  wie am Ende von Kapitel 1.3, und genau derselbe wie bei einem LLM, das eine Quelle erfindet
  und dabei völlig überzeugt klingt.
* **Die Verwechslungen sind nicht zufällig.** 7 und 9, 9 und 4, 2 und 8 — dieselben Paare, bei
  denen auch Menschen zögern. Das Netz hat etwas über die *Form* von Ziffern gelernt, nicht nur
  Pixel auswendig.

</details>

📖 Zum Schluss noch eine unbequeme Probe. Wir verschieben jedes Testbild um **4 Pixel nach
rechts** — für einen Menschen völlig egal.

▶️ Für das Netz nicht:

In [ ]:
verschoben = np.roll(bilder_test, 4, axis=2).reshape(-1, 784) / 255.0

fig, axes = plt.subplots(1, 6, figsize=(10, 2.2))
for k in range(3):
    axes[2 * k].imshow(bilder_test[k], cmap="gray")
    axes[2 * k].set_title("Original", fontsize=9)
    axes[2 * k].axis("off")
    axes[2 * k + 1].imshow(verschoben[k].reshape(28, 28), cmap="gray")
    axes[2 * k + 1].set_title("4 px nach rechts", fontsize=9)
    axes[2 * k + 1].axis("off")
plt.show()

print(f"Trefferquote auf den Originalbildern:   {treffer_test:.1%}")
print(f"Trefferquote auf den verschobenen:      {netz.score(verschoben, y_test):.1%}")

📖 **Von 97,7 % auf 37 %.** Das Netz hat nie gelernt, was eine 3 *ist* — es hat gelernt, welche
Pixel bei einer 3 typischerweise hell sind. Verschiebt man dieselbe 3 um vier Stellen, sind es
andere Pixel, und das Wissen ist weg.

Genau das behebt die nächste Modellgeneration: **Convolutional Neural Networks** schieben
denselben kleinen Mustererkenner über das ganze Bild und finden einen Bogen deshalb überall.
Auf MNIST kommen sie über 99,5 % — und sind gegen solche Verschiebungen weitgehend
unempfindlich.

---
## 9 · Das Netz mitnehmen

📖 Ein trainiertes Modell ist Arbeit — die will man nicht jedes Mal wiederholen. Also speichert
man die gelernten Zahlen auf die Festplatte und lädt sie später wieder.

Für `scikit-learn`-Modelle nimmt man `joblib`; bei Keras oder PyTorch heißt es `model.save()`
bzw. `torch.save()`. Der Inhalt ist immer derselbe: die Gewichte plus ein bisschen Beschreibung
der Architektur.

▶️ Speichern, neu laden, nachprüfen:

In [ ]:
import joblib

joblib.dump(netz, "ziffern_netz.joblib")
groesse = Path("ziffern_netz.joblib").stat().st_size / 1024**2
print(f"Gespeichert: ziffern_netz.joblib  ({groesse:.1f} MB)")

# … und in einer neuen Sitzung einfach wieder laden:
geladen = joblib.load("ziffern_netz.joblib")

print(f"Trefferquote des geladenen Netzes: {geladen.score(x_test, y_test):.2%}")
print(f"Identisch zum Original: {np.array_equal(geladen.predict(x_test), vorhersage_test)}")

📖 Gut 2 MB — davon sind knapp 900 KB die 109.386 gelernten Zahlen selbst (8 Byte je Zahl), der
Rest ist Zubehör aus dem Training.

Zum Vergleich: Ein LLM mit 70 Milliarden Parametern kommt in derselben Rechnung auf etwa
**140 GB** (dort rechnet man mit 2 Byte je Zahl). Es ist dieselbe Art Datei — eine Liste von
Zahlen — nur rund eine Million Mal länger.

---
## 10 · Was du gebaut hast

1. **Bilder als Zahlen verstanden** — 28 × 28 Pixel, normalisiert und flach gemacht.
2. **Verstanden, warum ein einzelnes Neuron nicht reicht** — und was Schichten daran ändern.
3. **ReLU und Softmax** selbst geschrieben — die beiden Bausteine, aus denen moderne Netze
   bestehen.
4. **Den Vorwärtsdurchlauf** selbst gebaut: drei Zeilen, die jedes neuronale Netz beschreiben.
5. **Ein Netz trainiert**, seine Trainingskurve gelesen und gesehen, was die erste Schicht lernt.
6. **Es angewendet** — mit deiner eigenen Funktion und den gelernten Gewichten.
7. **Ehrlich bewertet**: Testdaten, Konfusionsmatrix, Fehlerbilder, und die Grenzen des Ganzen.

Und jetzt der Blick zurück auf drei Kapitel:

| | Modellfunktion | Ausgabe | Kostenfunktion | Training |
|---|---|---|---|---|
| **Lineare Regression** (1.2) | $w \cdot x + b$ | eine Zahl | MSE | Gradientenabstieg |
| **Logistische Regression** (1.3) | $\sigma(w \cdot x + b)$ | eine Wahrscheinlichkeit | Log-Loss | Gradientenabstieg |
| **Neuronales Netz** (1.4) | $\text{Softmax}(\text{ReLU}(\dots) \cdot W_3 + b_3)$ | 10 Wahrscheinlichkeiten | Cross-Entropy | Gradientenabstieg |

---
# 🎉 Fertig!

Dein Netz liest handgeschriebene Ziffern besser, als du es je von Hand programmiert hättest.

### 🔬 Wer noch Zeit hat: dreh an den Stellschrauben

Die Zelle unten trainiert weitere Netze und vergleicht sie. Ein paar Fragen, die sich lohnen:

* Was macht **ein einziges Neuron** in der versteckten Schicht? Und was 512?
* Reichen **3 Epochen**? Was bringen 30?
* Was passiert **ganz ohne versteckte Schicht** — und warum ist das Ergebnis so vertraut?
* Was passiert, wenn du `activation="identity"` setzt, also **die ReLU ausbaust**?

*Achtung: Große Netze brauchen spürbar länger. Fang klein an.*

In [ ]:
# ▶️ Experimentierfeld — Architekturen ändern und vergleichen.
#    Zeilen ergänzen, ändern, löschen. Zum Beispiel:
#       {"hidden_layer_sizes": (), "max_iter": 15}                       ganz ohne Schicht
#       {"hidden_layer_sizes": (512, 256), "max_iter": 15}               deutlich größer
#       {"hidden_layer_sizes": (128, 64), "max_iter": 15,
#        "activation": "identity"}                                       ohne ReLU
VERSUCHE = [
    {"hidden_layer_sizes": (8,), "max_iter": 15},
    {"hidden_layer_sizes": (128,), "max_iter": 15},
    {"hidden_layer_sizes": (128, 64), "max_iter": 15},
]

print(f"{'Architektur':<20}{'Aktivierung':<14}{'Epochen':>9}{'Parameter':>12}{'Test':>9}")
print("-" * 64)
for einstellungen in VERSUCHE:
    versuch = MLPClassifier(random_state=42, early_stopping=True, batch_size=128,
                            **einstellungen)
    versuch.fit(x_train, y_train)
    zahlen = sum(w.size for w in versuch.coefs_) + sum(b.size for b in versuch.intercepts_)
    print(f"{str(einstellungen.get('hidden_layer_sizes', '(100,)')):<20}"
          f"{einstellungen.get('activation', 'relu'):<14}"
          f"{einstellungen.get('max_iter', 200):>9}{zahlen:>12,}"
          f"{versuch.score(x_test, y_test):>9.2%}")

### ☕ Oder einfach: Kaffee

Lehn dich zurück, während die anderen fertig werden. Und wenn du magst: schau mal nach links und
rechts. Jemandem beim Debuggen zu helfen ist die beste Art, selbst zu merken, ob man es
verstanden hat.